# Synthesis and Policy Implications

This notebook pulls together the strongest results from diagnostics, validation, and interpretation into a compact thesis-ready summary.

It is intended to answer three questions:
- What do the time-series diagnostics imply about PHP FX behavior?
- Why do hybrid models make sense statistically and economically?
- What should policymakers and risk managers take away?

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')
from statistical_validation import load_config, get_project_paths, discover_forecasts

config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
paths = get_project_paths(config)
forecasts = discover_forecasts(config)

eval_dir = paths['results_dir'] / 'evaluation'
diag_dir = paths['results_dir'] / 'diagnostics'
print('Target:', active_target)
print('Evaluation dir:', eval_dir)
print('Diagnostics dir:', diag_dir)

Target: PHP
Evaluation dir: results\PHP\evaluation
Diagnostics dir: results\PHP\diagnostics


#### Interpretation
This setup and load stage consolidates all key outputs used in final synthesis. If any source is missing here, policy conclusions should be treated as provisional.

In [2]:
metrics_path = eval_dir / 'metrics_summary.csv'
dm_path = eval_dir / 'dm_test_mse.csv'
stationarity_path = diag_dir / 'stationarity_panel.csv'
breaks_path = diag_dir / 'structural_breaks_panel.csv'

metrics_df = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
dm_df = pd.read_csv(dm_path) if dm_path.exists() else pd.DataFrame()
stationarity_df = pd.read_csv(stationarity_path) if stationarity_path.exists() else pd.DataFrame()
breaks_df = pd.read_csv(breaks_path) if breaks_path.exists() else pd.DataFrame()

summary = pd.DataFrame({
    'Metric': [
        'Test models discovered',
        'Pairs covered',
        'DM tests with p<0.05 share',
        'Stationarity rows available',
        'Structural-break rows available',
    ],
    'Value': [
        forecasts['Model'].nunique(),
        forecasts['Pair'].nunique(),
        float((dm_df['p_value'] < 0.05).mean()) if not dm_df.empty else np.nan,
        len(stationarity_df),
        len(breaks_df),
    ]
})
display(summary)

,Metric,Value
0,Test models discovered,9.000000
1,Pairs covered,5.000000
2,DM tests with p<0.05 share,0.618182
3,Stationarity rows available,10.000000
4,Structural-break rows available,5.000000


#### Interpretation
The summary table aggregates evidence from accuracy, diagnostics, and robustness. Read this as the final quantitative basis for selecting preferred model classes.

## Draft Policy and Research Implications

- For BSP and FX risk managers: the combination of stationarity, breaks, and volatility clustering means a static linear rule is insufficient; hybrid correction is preferable.
- For model builders: add macro exogenous variables such as US rate surprises, VIX, trade indicators, and remittance proxies to test ARIMAX/VARX variants.
- For thesis framing: the project is not only a forecast comparison; it is evidence that PHP FX is governed by both linear memory and nonlinear shock transmission.

In [3]:
fig = px.bar(
    summary,
    x='Metric',
    y='Value',
    title='Synthesis summary of key notebook outputs',
    template='plotly_white'
)
fig.show()

print('Policy note: when DM significance is high and break/volatility evidence is persistent, a hybrid residual model is justified for operational PHP FX monitoring.')

Policy note: when DM significance is high and break/volatility evidence is persistent, a hybrid residual model is justified for operational PHP FX monitoring.


#### Interpretation
This figure communicates implications for decision-making under volatility and spillovers. Policy or hedging recommendations should prioritize models that remain stable under stress.